# 08 Hawkes Processes and Market Impact

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/blob/main/09-Finance/08_Hawkes_Processes_and_Market_Impact.ipynb) [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/AmirrezaFarnamTaheri/Computational-Economics-and-Data-Science/main?filepath=09-Finance/08_Hawkes_Processes_and_Market_Impact.ipynb) [![Code License: MIT](https://img.shields.io/badge/Code%20License-MIT-yellow.svg)](../LICENSE) [![Content License: CC BY 4.0](https://img.shields.io/badge/Content%20License-CC%20BY%204.0-blue.svg)](https://creativecommons.org/licenses/by/4.0/)
## The Lens: Modeling Event Clustering Instead of Averaging It Away
High-frequency markets are event streams: orders, trades, cancellations, and quote changes arrive at irregular times. Their arrival intensity is not constant. A trade can trigger more trading, a burst of volatility can invite further activity, and feedback can produce clusters that a Poisson process cannot reproduce. A Hawkes process formalizes this self-excitation by letting every event temporarily raise future intensity.

The economic question is whether observed clustering reflects an endogenous feedback mechanism strong enough to matter for liquidity and execution. The crucial stability quantity is the branching ratio. When the expected number of descendants per event approaches one, activity becomes highly persistent and simulated paths can resemble cascades. We implement Ogata's thinning algorithm for an exponential Hawkes process, verify the stability condition, reconstruct the intensity path, and then connect event-flow risk to a separate empirical regularity: concave, approximately square-root market impact. The two objects should not be conflated—one models arrival dynamics, the other execution cost—but together they form a useful microstructure stress laboratory.

### Learning Objectives
- **Interpret** Hawkes intensity, excitation, decay, and branching ratio.
- **Implement** Ogata thinning for a stable exponential Hawkes process.
- **Reconstruct** event intensity and diagnose clustering.
- **Compare** event-flow feedback with a square-root market-impact benchmark.

### Prerequisites
- `06_High_Frequency_Data.ipynb`: market microstructure data and realized measures.
- `04_Continuous_Time_Finance.ipynb`: continuous-time stochastic processes.
- Point processes, exponential distributions, and Monte Carlo simulation.
* **Learning-path prerequisite:** [`07_Financial_Frictions_BGG.ipynb`](07_Financial_Frictions_BGG.ipynb)


> **Learning path:** Building on [`07_Financial_Frictions_BGG.ipynb`](07_Financial_Frictions_BGG.ipynb); this notebook closes the current track.


## Table of Contents

1. [Self-exciting point processes](#hawkes-model)
2. [Stability and branching](#stability)
3. [Ogata thinning](#ogata)
4. [Intensity diagnostics](#intensity-diagnostics)
5. [Square-root market impact](#market-impact)
6. [Exercises](#exercises)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.size": 12, "figure.figsize": (10, 6), "figure.dpi": 120})
np.set_printoptions(suppress=True, precision=5, linewidth=120)


<a id="hawkes-model"></a>
## 1. Self-Exciting Point Processes

For event times $t_i$, an exponential Hawkes intensity is

$$\lambda(t)=\mu+\sum_{t_i<t}\alpha e^{-\beta(t-t_i)},$$

where $\mu>0$ is baseline activity, $\alpha>0$ is the jump in intensity after an event, and $\beta>0$ controls decay. Conditional on the current history, the chance of an event in a short interval $dt$ is approximately $\lambda(t)dt$.


<a id="stability"></a>
## 2. Stability and Branching

The integral of one event's excitation kernel is

$$n=\int_0^\infty \alpha e^{-\beta s}ds=\frac{\alpha}{\beta}.$$

For a stationary univariate Hawkes process we require $n<1$. The long-run mean intensity is

$$E[\lambda(t)]=\frac{\mu}{1-n}.$$

Thus a seemingly small change in $n$ near one can have a large effect on average activity and clustering.


<a id="ogata"></a>
## 3. Ogata Thinning

Thinning samples candidate waiting times from an upper bound on the current intensity, advances the clock, recomputes the decayed intensity at the candidate time, and accepts with probability equal to the candidate intensity divided by the bound.


In [ ]:
def simulate_hawkes_exponential(mu=0.4, alpha=0.7, beta=1.2, horizon=200.0, seed=123):
    """Simulate a stable univariate exponential Hawkes process by Ogata thinning."""
    if min(mu, alpha, beta, horizon) <= 0:
        raise ValueError("Parameters and horizon must be positive.")
    if alpha >= beta:
        raise ValueError("Stationarity requires alpha / beta < 1.")
    local_rng = np.random.default_rng(seed)
    events = []
    t = 0.0
    while t < horizon:
        excitation_now = sum(alpha * np.exp(-beta * (t - ti)) for ti in events)
        upper = mu + excitation_now
        t += local_rng.exponential(1.0 / upper)
        if t >= horizon:
            break
        intensity = mu + sum(alpha * np.exp(-beta * (t - ti)) for ti in events)
        if local_rng.random() <= intensity / upper:
            events.append(t)
    return np.asarray(events)

mu, alpha, beta, horizon = 0.4, 0.7, 1.2, 200.0
events = simulate_hawkes_exponential(mu, alpha, beta, horizon)
branching_ratio = alpha / beta
empirical_rate = len(events) / horizon
theoretical_rate = mu / (1 - branching_ratio)
print(f"events={len(events)}, branching ratio={branching_ratio:.3f}")
print(f"empirical rate={empirical_rate:.3f}, stationary mean rate={theoretical_rate:.3f}")
assert branching_ratio < 1


<a id="intensity-diagnostics"></a>
## 4. Intensity Diagnostics

A single path is noisy, so agreement with the theoretical stationary rate is only approximate. More informative diagnostics compare inter-arrival distributions, count dispersion across equal time bins, and behavior across many simulated paths. A Poisson process has count variance approximately equal to its mean; self-excitation generally produces over-dispersion.


In [ ]:
grid = np.linspace(0, horizon, 2_000)
intensity = np.full_like(grid, mu)
for ti in events:
    mask = grid > ti
    intensity[mask] += alpha * np.exp(-beta * (grid[mask] - ti))

counts, edges = np.histogram(events, bins=40, range=(0, horizon))
dispersion = counts.var(ddof=1) / counts.mean()
print(f"count variance/mean ratio = {dispersion:.3f}")

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(grid, intensity)
axes[0].vlines(events, 0, mu * 0.35, alpha=0.25, lw=0.8)
axes[0].set(ylabel="intensity", title="Self-exciting order-flow intensity")
axes[1].bar(edges[:-1], counts, width=np.diff(edges), align="edge")
axes[1].set(xlabel="time", ylabel="events per bin")
plt.show()


<a id="market-impact"></a>
## 5. Square-Root Market Impact

A widely used empirical benchmark for the price impact of executing quantity $Q$ against daily volume $V$ is

$$I(Q)=Y\sigma\sqrt{\frac{Q}{V}},$$

where $\sigma$ is a volatility scale and $Y$ is an order-one coefficient estimated for the relevant market. This is a reduced-form execution-cost relation, not a consequence of the Hawkes model. It is useful here because a burst of self-excited order flow can move an execution into a different participation-rate regime.


In [ ]:
def square_root_impact(quantity, daily_volume, volatility=0.02, y_coefficient=0.8):
    quantity = np.asarray(quantity, dtype=float)
    if np.any(quantity < 0) or daily_volume <= 0 or volatility < 0 or y_coefficient < 0:
        raise ValueError("Impact inputs must be nonnegative and volume positive.")
    return y_coefficient * volatility * np.sqrt(quantity / daily_volume)

participation = np.logspace(-4, -0.3, 100)
impact = square_root_impact(participation, 1.0)
fig, ax = plt.subplots()
ax.loglog(participation, impact)
ax.set(xlabel="Q / V", ylabel="fractional impact", title="Concave square-root market-impact benchmark")
plt.show()


## Key Equations

These relations are collected from the derivations above as a review map. Their assumptions and derivations remain part of the result; this box is not a substitute for them.

**1. Core relation**

$$\lambda(t)=\mu+\sum_{t_i<t}\alpha e^{-\beta(t-t_i)},$$

**2. Core relation**

$$n=\int_0^\infty \alpha e^{-\beta s}ds=\frac{\alpha}{\beta}.$$

**3. Core relation**

$$E[\lambda(t)]=\frac{\mu}{1-n}.$$

**4. Core relation**

$$I(Q)=Y\sigma\sqrt{\frac{Q}{V}},$$


## Exercises

**1. Stability (Conceptual):** Derive the stationary mean intensity from the immigrant-offspring interpretation. Why does it diverge as `alpha/beta → 1`?

**2. Clustering (Applied):** Simulate 200 paths for branching ratios 0.1, 0.5, 0.8, and 0.95 while holding the theoretical mean event rate fixed. Compare count dispersion and maximum local intensity.

**3. Bivariate microstructure (Challenge):** Extend the simulator to mutually exciting buy/sell processes with a 2×2 excitation matrix. State the spectral-radius stability condition and examine how cross-excitation changes order-sign autocorrelation.


## Summary & Key Takeaways

- Hawkes processes model endogenous event clustering through history-dependent intensity.
- The branching ratio `alpha/beta` is both an interpretable feedback measure and the core univariate stationarity diagnostic.
- Ogata thinning provides an exact simulation route for the exponential-kernel specification.
- Market impact is a distinct reduced-form object; joining it to event-flow dynamics is a stress-analysis exercise, not an identity.


## References & Further Reading

- Hawkes, A. G. (1971). Spectra of some self-exciting and mutually exciting point processes. *Biometrika*, 58(1), 83–90.
- Ogata, Y. (1981). On Lewis' simulation method for point processes. *IEEE Transactions on Information Theory*, 27(1), 23–31.
- Bacry, E., Mastromatteo, I. & Muzy, J.-F. (2015). Hawkes processes in finance. *Market Microstructure and Liquidity*, 1(1), 1550005.
